<a href="https://colab.research.google.com/github/kavy988/ai-ml-dl-learning/blob/main/deep-learning/04_pytorch_custom_datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

check out the datasets on pytorch

# pytorch custom datasets video notebook

we've used some datasets with pytorch before

but how dou get your own data into pytorch

one of the ways to do so id via : custom datastes.

## Domain libraries

depending on what you are working on , vision, text , audio , recommendation , you ll want to look into each of the pytorch domain libraries for existing data loading fucntions and customaziable data loading fucntions .


https://docs.pytorch.org/vision/stable/datasets.html

** Resources **
* book version of the course materials at learnpytorch


# 0.  importing pytorch and setting up device-agnostic code

In [ ]:
import torch
from torch import nn

# note :  pytorch 2.11.0+cpu
torch.__version__

In [ ]:
# setup device-agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
device


# 1. Get data

https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.Food101.html?highlight=food101


* our dataset is a subset of the Food101 dataset

Food101 starts 101 different classes of food and 1000 images per class (750 training , 250 testing ).

our dataset starts with 3 classes of food and only 10% of the images (-75 trainign , 25 testing).

why do this ?
 when starting out ML projects its improtant to try things on a small scale and then increase the scale when necessary . the whole point is to speed up up how fast uou can experiment .

https://data.vision.ee.ethz.ch/cvl/datasets_extra/food-101/

* the Food101 dataset

In [ ]:
# import requests          # web se file download karne ke liye
# import zipfile           # zip file ko extract (unzip) karne ke liye
# from pathlib import Path # folder/file paths handle karne ke liye

# data folder ka path set kiya
# data_path = Path("data/")
# image_path = data_path / "pizza_steak_sushi"  # data/pizza_steak_sushi wala path

# check karo ki folder already hai ya nahi
# if image_path.is_dir():
  # print(f"{image_path} dirextory already exists ... skipping one...")
# else :
  # print(f"{image_path} does not exist , creating pne ...")
  # image_path.mkdir(parents = True , exist_ok = True )  # folder bana do (parent folders bhi agar missing ho)


# zip file ko internet se download karke save karo
# with open(data_path / "pizza_steak_sushi.zip" , "wb") as f: #wb stands for write binary  and r stands for read

  # request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/data/pizza_steak_sushi.zip")  # GitHub se zip file mangwao
  # print("downloading pizza ,steak , sushi data...")
  # f.write(request.content)  # download hui content ko file mein likh do

# downloaded zip ko khol (extract) karo
# with zipfile.ZipFile(data_path / "pizza_steak_sushi.zip" , "r") as zip_ref:
  # print("unzipping pizaa , steak and sushi data ...")
  # zip_ref.extractall(image_path)  # sab files image_path folder mein nikaal do

# TBR

* is_dir()  - check karta hai ki given path ek folder (directory/ Folder ) hai ya nahi — aur exist karta hai ya nahi.

* directory = folder. Same cheez hai, bas directory zyada technical/programming term hai.

In [ ]:
import requests
import zipfile
from pathlib import Path

# setup path to a data folder
data_path = Path("data/")
image_path = data_path / "pizza_steak_sushi"

# if the image folder doesnt exist , download it and prepare it
if image_path.is_dir():
  print(f"{image_path} dirextory already exists ... skipping one...")
else :
  print(f"{image_path} does not exist , creating pne ...")
  image_path.mkdir(parents = True , exist_ok = True )


# Download
with open(data_path / "pizza_steak_sushi.zip" , "wb") as f: #wb stands for write binary  and r stands for read

  request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/data/pizza_steak_sushi.zip")
  print("downloading pizza ,steak , sushi data...")
  f.write(request.content)

# unzip pizza ,  steak , sushi data
with zipfile.ZipFile(data_path / "pizza_steak_sushi.zip" , "r") as zip_ref:
  print("unzipping pizaa , steak and sushi data ...")
  zip_ref.extractall(image_path)


In [ ]:
image_path

# 2. Becoming one with the data (data preparation and data exploration )

In [ ]:
import os
def walk_through_dir(dir_path):
  """Walks through dir_path returning its contents"""
  for dirpath , dirnames , filenames in os.walk(dir_path):
    print(f" there are {len(dirnames)} directories and {len(filenames)} images  in '{dirpath}' . ")



In [ ]:
walk_through_dir(image_path)

In [ ]:
import os
for item in os.listdir(image_path):
    print(item)


In [ ]:
for item in os.listdir(image_path / "train"):
    print(item)


In [ ]:
# setup train and testing path
train_dir  = image_path / "train"
test_dir = image_path / 'test'

train_dir , test_dir

# TBR

* (PosixPath) -  ek object type hai jo pathlib.Path folder/file paths ko represent karne ke liye deta hai (Mac/Linux systems pe).

### 2.1 Visualizing and image

* lets write some code to :
1. Get all of the image paths
2. pick a random image path using python random choice()
3. get the image class name using 'pathlib.Path.parent.stem'
4. since we are working wiht images , lets open the image python's PIL.
5. we'll then show the image and print metadata .

# TBR

* PIL = Python Imaging Library

-> Yeh Python ka library hai images open, edit, aur process karne ke liye. (Aajkal iska maintained version Pillow hai, lekin import karte time still from PIL import Image likhte hain.)

In [ ]:
import random # image random choose karne ke leya
from PIL import Image # image open/read karna ke leya

# set seed
# random.seed(42)

# 1. get all image paths
image_path_list = list (image_path.glob("*/*/*.jpg"))
# glob("*/*/*.jpg") matlab 3 folder levels ke andar jitni bhi .jpg files hain, sab uthao

# 2. pick a random image path
random_image_path  = random.choice(image_path_list)
print(random_image_path)

#3. get image class from path name (the image class is the name of the directory where the image is stored )
image_class = random_image_path.parent.stem
# .parent = image ke upar wala folder, .stem = uska naam (bina extension)
print(image_class)

# 4. open image
img = Image.open(random_image_path)

# 5. print metadata
print(f"random image path : {random_image_path}")
print(f"Image class : {image_class}")
print(f"Image height : {img.height}")
print(f"Image width : {img.width}")
img

In [ ]:
# try to visualize the imge with matplotlib
import numpy as np
import matplotlib.pyplot as plt

# turn the image into an array
img_as_array = np.asarray(img)

# plit the image with matplotlib
plt.figure(figsize =(10,7))
plt.imshow(img_as_array)
plt.title(f"image class : {image_class} | image shape : {img_as_array.shape} -> [height, width , color_channels] , [hwc]")
plt.axis(False)


In [ ]:
img_as_array

## 3. transformign data

* before we can use our image data with pytorch:

1. turn your target data into tensors (in our case , its gonna be numerical representation of our images.)

2. turn it into a 'torch.utils.data.Dataset' ans subsequently a 'torch.utils.data.DataLoader'


In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets , transforms



### 3.1 transforming data with torchvision.transforms

transforms help you get your images ready to be used with a model/perform data segmentation

https://docs.pytorch.org/vision/0.22/transforms.html

In [ ]:
# write a transforms for image
data_transform = transforms.Compose([
    # Reasize our images to 64*64
    transforms.Resize(size =(64,64)),

    # flip the images randomly on horizontal
    transforms.RandomHorizontalFlip(p = 0.5),
    # p = 0.5 matlab 50% chance hai ki image horizontally flip (mirror) ho jayegi

    # turn the image into torch tensor
    transforms.ToTensor()
])


In [ ]:
data_transform(img).shape

In [ ]:
data_transform(img).dtype

# TBR

* k = fixed naam (library ne diya), uski value = kitni photos chahiye (yaha n=3 diya hai).

In [ ]:
def plot_transformed_images(image_paths, transform , n=3, seed =42):
  """ selects random images from a path of images and loades/transforms
  them then plots the original vs the transformed version """

  if seed:
    random.seed(seed)
  random_image_paths = random.sample(image_paths, k = n)

  for image_path in random_image_paths:
    with Image.open(image_path) as f:
      fig, ax = plt.subplots(nrows = 1, ncols = 2)
      ax[0].imshow(f)
      ax[0].set_title(f"priginal\nSize : {f.size}")
      ax[0].axis(False)

      # transforms and plot target image
      transformed_image = transform(f).permute(1, 2, 0) # note we will need to the change shape for matplolib (C , H , W) -> (H ,W, C)
      ax[1].imshow(transformed_image)
      ax[1].set_title(f"Transformed\nSize ; {transformed_image.shape}" )
      ax[1].axis(False)

      fig.suptitle(f"Class : {image_path.parent.stem}" , fontsize = 16)

plot_transformed_images(image_paths = image_path_list,
                        transform = data_transform,
                        n =3,
                        seed = 42)

https://docs.pytorch.org/vision/main/auto_examples/transforms/plot_transforms_illustrations.html

* check this out


## 4. optiion 1: loading image data using

https://docs.pytorch.org/vision/main/generated/torchvision.datasets.ImageFolder.html


* we can load image classfication data using torchvision.datasets.ImageFolder

In [ ]:
# use ImageFolder to create dataste(s)

from torchvision import datasets
train_data = datasets.ImageFolder(root = train_dir,
                                  transform = data_transform , # a transform for the data
                                  target_transform = None  # a transform for the labels/ target
                                  )
test_data = datasets.ImageFolder(root = test_dir,
                                 transform = data_transform)

train_data, test_data

In [ ]:
train_dir , test_dir

In [ ]:
# get class names as list
class_name = train_data.classes
class_name

In [ ]:
# get class names as dict
class_dict = train_data.class_to_idx
class_dict

In [ ]:
# check the lengths of our dataset
len(train_data ) , len(test_data)


In [ ]:
train_data.targets[:20]

In [ ]:
train_data.samples[0]

# TBR

* train_data.samples

-> yeh ImageFolder ka ek attribute hai jo saari images ke paths + unke labels ki list deta hai.

OR

-> Short mein: yeh tuple hai — (image path, uska class number).


* train_data.targets

-> har image ke liye uska class number, order mein

* type(label) kaam karta hai kyunki type() ek Python built-in function hai jo kisi bhi object ka type bata sakta hai — yeh normal int, str, list, kuch bhi ho, sab pe kaam karta hai.

* label.type(torch.float32) fail hua kyunki .type() yeh koi Python built-in nahi hai — yeh PyTorch tensor ka method hai (sirf tensors pe kaam karta hai, dtype convert karne ke liye).

* label abhi ek plain Python int hai (tensor nahi), isliye usme .type() method exist hi nahi karta — isi wajah se error aaya: 'int' object has no attribute 'type'.

In [ ]:
# index on the train_data dataset to get a single image and label
img, label = train_data[0][0] , train_data[0][1]
print(f"imgage tensor :\n {img}")
print(f"Image shape : {img.shape}")
print(f"Image datatype : {img.dtype}")
print(f"Image label : {label}")
print(f"Label datatype. : {type(label)}")

In [ ]:
class_name[label]

In [ ]:
# Rearrange the order dimension
img_permute = img.permute(1,2,0)

# print out different shapes
print(f" original shap : {img.shape} -> [color_channels , height ,width]  ")
print(f"image permute : {img_permute.shape} -> height , width , color_channels")

# plot the image
plt.figure(figsize = (10, 7))
plt.imshow(img_permute)
plt.axis("off")
plt.title(class_name[label] , fontsize = 14)

### 4.1 turn loaded images into DataLoader

A 'dataloader' is going to help us turn our 'dataset'. into iterables and we can customize the batch_size so our model can see the batch_size image at the same time


In [ ]:
# turn our  train and test datasets into DataLoader's

from torch.utils.data import DataLoader
BATCH_SIZE = 1
train_dataloader = DataLoader(dataset = train_data,
                              batch_size =  BATCH_SIZE,
                              num_workers = 1,
                              shuffle = True)

test_dataloader = DataLoader(dataset = test_data,
                             batch_size = BATCH_SIZE,
                             num_workers = 1,
                             shuffle = False)

train_dataloader ,test_dataloader


In [ ]:
len(train_dataloader) , len(test_dataloader)

In [ ]:
img, label = next(iter(train_dataloader))


# batch size will be now 1 , you can change the batch size if u like
print(f" Image shape : {img.shape} -> [batch_size , color_channels ,height , width] ")
print(f" Label shape : {label.shape}")

## 5 optin 2. : Loading image data with a custom 'Dataset'

1. want to be able to load images from file
2. want to be able to get class names from the datset
3. want to be able to get classes as dictionary from the datset.

pros:
* can create a 'dataset' out of almost anythibg
* not limited to pytorch prebuilt 'dataset' functions

cons:
* even though you could create 'dataset' out of almost anything , it doesnt mean it will work...
* using a custom 'dataset' often results in us writing more code , which could be prone to errors to permformance issues


All custom datasets in pytorch often subclass - https://docs.pytorch.org/vision/stable/datasets.html

# torch.utils.data.Datset

In [ ]:
import os
import pathlib
import torch

from PIL import Image
from torch.utils.data import Dataset
from typing import Tuple , Dict , List



In [ ]:
# instance of torchvision.datsets.ImageFolder()
train_data.classes , train_data.class_to_idx

### 5.1 creating a helper function to get class names

we want a function to :

1. get the lcass names using 'os.scandir()'to tranverse a target directory (ideally the directory is in standard imgae classification format ).

2. Raise an error if the class names arent found (if this happens , there might be something wrong with the directory structure .

3. turn the class names into a dict and a list and return them. .

* os.scandir() - ek folder ke andar jo bhi files/folders hain, unki list dikhata hai — jaise os.listdir(), lekin thoda fast aur zyada info ke saath (jaise woh entry file hai ya folder, bina extra check kiye).

In [ ]:
# setup path for target directory
target_directory = train_dir  # folder ka path set kiya (train wala)
print(f"Target dir : {target_directory}")

# get the class names from the target directory
class_name_found = sorted([entry.name for entry in list(os.scandir(target_directory))])
class_name_found
# os.scandir() se saari entries mili, .name se sirf naam liya, sorted() se alphabetically arrange kiya

In [ ]:
list(os.scandir(target_directory))

In [ ]:
def find_classes(directory: str) -> Tuple[list[str] , Dict[str, int] ] :
  """ Find  the class folder names in a target directory ."""
  # 1. get the class names by scanning the target direcotry
  classes  = sorted(entry.name for entry in os.scandir(directory) if entry.is_dir())

  # 2. Raise an error if class names could not be foudn
  if not classes:
    raise FileNotFoundError(f"couldnt find any classes in {directory}..... please check hte file structure.")

  # 3. create a directory of index labels(computers prefer numvbers rather than strings as labels)
  class_to_idx = {class_name: i for i, class_name in enumerate(classes)}
  return classes , class_to_idx



In [ ]:
find_classes(target_directory)

### 5.2 create a custom 'Dataset' to replicate the 'ImageFolder'

to create our own custom dataset , we wamt to :   
1. SUbclass 'torch.utils.data.Dataset'.
2. init our subclass with a target directory (the directory we'd like to get data from) as well as a transform if we'd like to transform our data .
3. Create several attributres :

* paths - paths of our images.

* transforms - the transform we'd like to use .

* classes - a list of the target classes

* class_to_idx - a dict of the target classes mapped to integer labels

4. create a function to 'load_images()' , this function will open an image

5. overwrite the '__len()__' method to return the length of our dataset

6. overwrtie the __getitem()__ method to return a given sample when passed an index



In [ ]:
# write a custom dataset class
from torch.utils.data import Dataset

# 1. subclass torch.utils.data.Dataset
class ImageFolderCustom(Dataset):
  # 2. Inititalize our custom dataset
  def __init__(self,
               targ_dir: str,
               transform = None):
    # 3. create class attributes
    # get all of the image paths
    self.paths = list(pathlib.Path(targ_dir).glob("*/*.jpg"))
    # setup transforms
    self.transforms = transform
    # create classes and class_to_idx attributes
    self.classes , self.class_to_idx = find_classes(targ_dir)


  # 4. create a function to load images
  def load_images(self, index: int ) -> Image.Image:
    "opens an image via a pth and returns it ."
    image_path = self.paths[index]
    return Image.open(image_path)

  # 5. overwrite __len__()
  def __len__(self) -> int:
    "returns the total number of samples ."
    return len(self.paths)


  # 6. overwrite __getitem__()  method to return a particular sample
  def __getitem__(self, index: int) -> Tuple[torch.Tensor, int]:
    "Returns one sample pf data , data and label (X,y). "
    img = self.load_images(index)
    class_name = self.paths[index].parent.name # expects path in format: data_folder/class_name/image.jpg
    class_idx = self.class_to_idx[class_name]


    # transform if necessary
    if self.transforms:
      return self.transforms(img) , class_idx # return data, label (X,y)
    else :
      return img, class_idx # return untransformed image and label

In [ ]:
# create a transform
train_transforms = transforms.Compose([
    transforms.Resize(size = (64, 64)),
    transforms.RandomHorizontalFlip(p = 0.5),
    transforms.ToTensor()
])

test_transforms = transforms.Compose([
    transforms.Resize(size= (64, 64)),
    transforms.ToTensor()
])


In [ ]:
# test out ImageFolderCustom
train_data_custom = ImageFolderCustom(targ_dir = train_dir,
                                      transform = train_transforms)
test_data_custom = ImageFolderCustom(targ_dir = test_dir,
                                     transform = test_transforms)


In [ ]:
len(train_data), len(train_data_custom)

In [ ]:
len(test_data) , len(test_data_custom)

In [ ]:
train_data_custom.classes

In [ ]:
train_data_custom.class_to_idx

In [ ]:
# check for equality between original ImageFolder Dataset and ImageFolderCustomDataset
print(train_data_custom.classes == train_data.classes)
print(test_data_custom.classes == test_data.classes)

### 5.3 creat a cuntion to display random images

1. take in a 'Datase' and a no. of other parameters such as class names and how many images to visualize .
2. to prevent the display getting out of hand , lets cap the number of images to see at 10 .
3. set the random seed for reproducibility .
4. Get a list of random sample indexes from the tagret dataset.
5. setup a matplotlib plot.
6. loop through the random sample images and plot them with maptlotlib
7. make sure the dimensions of our images line up wiht matplotlib(HWC).

In [ ]:
# 1. create a function to take in daatset
def display_random_images (dataset: torch.utils.data.Dataset,
                           classes : List[str] = None,
                           n : int = 10 ,
                           display_shape : bool  = True,
                           seed: int = None):
  # 2. adjust display if n is to high
  if n > 10:
    n = 10
    display_shape = False
    print(f"For display , purpose , n should'nt be larger than 10 , setting to 10 and removing shape display ")

  # 3. set the seed
  if seed:
    random.seed(42)

  # 4. Get random sample indexes
  random_sample_idx = random.sample(range(len(dataset)) , k =n )

  # 5. setup plot
  plt.figure(figsize =(16 , 8))


  # 6. loop through random indexes and plot them with matplot
  for i , targ_sample in enumerate(random_sample_idx):
    targ_image, targ_label = dataset[targ_sample][0], dataset[targ_sample][1]

    # 7. Adjust tensor dimensions for plotting
    targ_image_adjsut = targ_image.permute(1,2,0) # [color_channels , height ,width] -> [WHC]

    # plot adjusted samples
    plt.subplot(1 ,n , i+1)
    plt.imshow(targ_image_adjsut)
    plt.axis("off")
    if classes:
      title = f"Class : {classes[targ_label]}"
      if display_shape:
        title = title + f"\nshape : {targ_image_adjsut.shape}  "
    plt.title(title)

In [ ]:
# display random images  the ImageFolder   created dataste
display_random_images(train_data,
                      n = 5,
                      classes = class_name,
                      seed = 42)

In [ ]:
# display random images from the ImageFolderCustom Dataset
display_random_images(train_data_custom,
                      n = 20,
                      classes = class_name,
                      seed = None)

### 5.4 Turn custom loaded images into DataLoader's


In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32
NUM_WORKERS = os.cpu_count()
train_dataloader_custom = DataLoader(dataset = train_data_custom,
                                     batch_size =BATCH_SIZE,
                                     num_workers = NUM_WORKERS,
                                     shuffle = True)

test_dataloader_custom = DataLoader(dataset = test_data_custom,
                                    batch_size = BATCH_SIZE,
                                    num_workers = NUM_WORKERS,
                                    shuffle = False)

train_dataloader_custom, test_data

In [ ]:
len(train_dataloader_custom) , len(test_dataloader_custom)

In [ ]:
# get image and label from custom dataloader
img_custom , label_custom = next(iter(train_dataloader_custom))

# print out the shpae
img_custom.shape , label_custom.shape


## 6. other forms of transforms (data_augementation)

* data augmentation is the process of artificially adding diversity to ur training data .

* In the case of image data , this may mean appplying various image transformation to training images.

this practice hopefully results in a model that is generally an uness data

* lets take a look at one particular type of data augmnetation used to train pytorch vision models to state of the art levels..

https://pytorch.org/blog/how-to-train-state-of-the-art-models-using-torchvision-latest-primitives/

In [ ]:
# lets look at trivialaugment https://pytorch.org/blog/how-to-train-state-of-the-art-models-using-torchvision-latest-primitives/

from torchvision import transforms
train_transform = transforms.Compose([
    transforms.Resize(size =(224, 224)),
    transforms.TrivialAugmentWide(num_magnitude_bins = 31),
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.Resize(size= (224, 224)),
    transforms.ToTensor()
])

In [ ]:
# get all images paths
image_path_list = list(image_path.glob("*/*/*.jpg"))
image_path_list[:10]


In [ ]:
# plot random transformed images
plot_transformed_images(
    image_paths = image_path_list,
    transform = train_transform,
    n=3,
    seed = None
)

## 7. model_0 :  TinyVGG withough data augmentation


* https://poloclub.github.io/cnn-explainer/


lets replicate TinyVGG architecture from the CNN explainer wesbite

### 7.1 creating transforms and loadin data for model_0

In [ ]:
# create simple transforms
simple_transform = transforms.Compose([
    transforms.Resize(size = (64, 64)),
    transforms.ToTensor()

])



In [ ]:
# 1. load and transform data
from torchvision import datasets
train_data_simple = datasets.ImageFolder(root = train_dir,
                                         transform = simple_transform)

test_data_simple = datasets.ImageFolder(root = test_dir,
                                        transform = simple_transform)


# 2. Turn the datasets into DataLoaders
import os
from torch.utils.data import DataLoader

# setup batch size and number of works
BATCH_SIZE = 32
NUM_WORKERS = os.cpu_count()

# create dataloader's
train_dataloader_simple = DataLoader(dataset = train_data_simple,
                                     batch_size = BATCH_SIZE,
                                     shuffle = True,
                                     num_workers = NUM_WORKERS)

test_dataloader_simple = DataLoader(dataset = test_data_simple,
                                    batch_size = BATCH_SIZE,
                                    shuffle = False,
                                    num_workers = NUM_WORKERS)


In [ ]:
NUM_WORKERS

### 7.2 create TinyVGG model class

In [ ]:
class TinyVGG(nn.Module):
  """
  model architecture copying TinyVGG from cnn Explainer
  """
  def __init__(self ,
               input_shape : int,
               hidden_units : int ,
               output_shape : int
               ) -> None :

    super() .__init__()
    self.conv_block_1 = nn.Sequential(
        nn.Conv2d(in_channels = input_shape,
                  out_channels = hidden_units,
                  kernel_size = 3,
                  stride = 1,
                  padding = 0),
        nn.ReLU(),
        nn.Conv2d(in_channels = hidden_units,
                  out_channels = hidden_units,
                  kernel_size = 3,
                  stride = 1,
                  padding = 0),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2,
                     stride = 2)# default stride value is same kernel size
    )
    self.conv_block_2 = nn.Sequential(
        nn.Conv2d(in_channels = hidden_units,
                  out_channels = hidden_units,
                  kernel_size = 3,
                  stride = 1,
                  padding = 0),
        nn.ReLU(),
        nn.Conv2d(in_channels = hidden_units,
                  out_channels = hidden_units,
                  kernel_size = 3,
                  stride = 1,
                  padding = 0),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2,
                     stride = 2)# default stride value is same kernel size
    )
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features = hidden_units*13*13,
                  out_features = output_shape)
    )
  def forward(self , x):
    x = self.conv_block_1(x)
    # print(x.shape)
    x = self.conv_block_2(x)
    # print(x.shape)
    x = self.classifier(x)
    # print(x.shape)
    return x
    return self.classifier(self.conv_block_2(self.conv_block_1(x))) # benifits from operator fusion


https://horace.io/brrr_intro.html how to make ur cpu go brr or fast

In [ ]:
torch.manual_seed(42)
model_0 = TinyVGG(input_shape = 3, # no. of color channels in our image data
                  hidden_units = 10,
                  output_shape = len(class_name)).to(device)

model_0


### 7.3 Try a forward pass on a single image(to test the model)


In [ ]:
# get a single image batch
image_batch, label_batch = next(iter(train_dataloader_simple))
image_batch.shape, label_batch.shape


In [ ]:
# Try a forward pass
model_0(image_batch.to(device))
# model_0(image_batch)


### 7.4 Use 'torchinfo' to get an idea of the shapes going through our model

In [ ]:
# Install torchinfo , import if its available
try:
  import torchinfo
except:
  !pip install torchinfo
  import torchinfo

from torchinfo import summary
summary (model_0 ,input_size =[1,3,64,64 ])

## 7.5 Create train and test loops function

* 'train_step()' - takes in a model and dataloader and trains the model on the dataloader.
* 'test_step()' -  takes in a model and dataloader and evaluates the model on the dataloader.

In [ ]:
# Create train_step()
def train_step(model: torch.nn.Module,
               dataloader : torch.nn.Module,
               loss_fn : torch.nn.Module,
               optimizer : torch.optim.Optimizer,
               device = device):

  # put the model in train mode
  model.train()

  # setup train loss and train accuracy values
  train_loss , train_acc = 0,0

  # loop through data loade data batches
  for batch ,(X,y) in enumerate(dataloader):
    # send data to the target device
    X, y = X.to(device) ,y.to(device)

    # 1. Forward pass
    y_pred = model(X)

    # 2. Calculate the loss
    loss = loss_fn(y_pred , y)
    train_loss += loss.item()

    # optimizer zero grad
    optimizer.zero_grad()

    # 4. loss backward
    loss.backward()

    # 5.optimzier step
    optimizer.step()

    # claculate accuracy metric
    y_pred_class = torch.argmax(torch.softmax(y_pred , dim = 1), dim = 1)
    train_acc += (y_pred_class == y).sum().item()/len(y_pred)


  # adjust metrics to get average loss and accuracy per batch
  train_loss = train_loss / len(dataloader)
  train_acc =  train_acc / len(dataloader)
  return train_loss , train_acc



In [ ]:
# create  a test step
def test_step(model : torch.nn.Module,
              dataloader : torch.utils.data.DataLoader,
              loss_fn : torch.nn.Module,
              device = device):

  # put model in eval mode
  model.eval()

  # setup test loss and test accuracy values
  test_loss , test_acc = 0,0

  # Turn on inference mode
  with torch.inference_mode():
    # Loop through DataLoader batches
    for batch , (X, y) in enumerate (dataloader):
      # Send data to the target device
      X, y = X.to(device), y.to(device)

      # 1.Forward pass
      test_pred_logits = model(X)

      # 2. ca;culate the loss
      loss = loss_fn(test_pred_logits , y)
      test_loss += loss.item()

      # 3. Calculate the accuracy
      test_pred_labels = test_pred_logits.argmax(dim = 1)
      test_acc += ((test_pred_labels == y).sum().item() / len(test_pred_labels))

  # Adjust metrics to get averagae loss and accuracy per batch
  test_loss = test_loss / len(dataloader)
  test_acc  = test_acc / len(dataloader)
  return test_loss , test_acc


### 7.6 Creating a 'train()' function to combine train_step() and test_step()



In [ ]:
from tqdm.auto import tqdm

# 1. creata a train fn that takes in various model parameters + optimizer + dataloader + loss_function

def train(model : torch.nn.Module,
          train_dataloader : torch.utils.data.DataLoader,
          test_dataloader : torch.utils.data.DataLoader,
          optimizer : torch.optim.Optimizer,
          loss_fn : torch.nn.Module = nn.CrossEntropyLoss(),
          epochs : int = 5,
          device = device):
  # 2. Create empty results dictonary
  results = {"train_loss" : [],
             "train_acc" : [],
             "test_loss" : [],
             "test_acc" : []}

  # 3. Loop through training and testing steps for a number of epochs
  for epoch in tqdm(range(epochs)):
    train_loss , train_acc = train_step(
        model = model,
        dataloader = train_dataloader,
        loss_fn = loss_fn,
        optimizer = optimizer,
        device = device
    )

    test_loss , test_acc = test_step(model = model,
                                     dataloader = test_dataloader,
                                     loss_fn = loss_fn,
                                     device = device)

    # 4. print out whats happening
    print(f"Epochs : {epoch} | train loss : {train_loss:.4f} , train acc : {train_acc:.4f} | Test loss : {test_loss:.4f} | test acc : {test_acc:.4f}")


    # 5. Update results dictionary
    results["train_loss"].append(train_loss)
    results["train_acc"].append(train_acc)
    results["test_loss"].append(test_loss)
    results["test_acc"].append(test_acc)

  # 6. Return the filled reuslts at the end of the epoch
  return results


### 7.7 Train and evaluate model 0


In [ ]:
# set random seeds
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# set no.of epochs
NUM_EPOCHS = 5

# Recreate an instance of TinyVGG
model_0 = TinyVGG(input_shape = 3,
                  hidden_units = 10,
                  output_shape = len(train_data.classes)).to(device)

# setup loss function and optimzier
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params = model_0.parameters(),
                             lr = 0.001)

#  start the timer
from timeit import default_timer as timer
start_time = timer()

# train model_0
model_0_results = train(model = model_0,
                        train_dataloader = train_dataloader_simple,
                        test_dataloader = test_dataloader_simple,
                        optimizer = optimizer,
                        loss_fn = loss_fn ,
                        epochs = NUM_EPOCHS)

# End the timer and print out how long it took
end_time = timer()
print(f"Total training time : {end_time-start_time:.3f} seconds")

In [ ]:
model_0_results

## 7.8 Plot the loss curves of Modle 0

A loss curve is a way of tracking your model's progress over time .
agood guide for different loos curves be seen here  - https://developers.google.com/machine-learning/crash-course/overfitting/interpreting-loss-curves

In [ ]:
# GEt the model_0_results keys
model_0_results.keys()


In [ ]:
def plot_loss_curves(results : Dict [str , List[float] ]):
  """ Plots training curves of a results dictionary"""
  # Get the loss values of the results dictionary (training and test)
  loss = results["train_loss"]
  test_loss = results["test_loss"]


  # Get the accuracy values of the results dictionary (training and test):
  accuracy = results["train_acc"]
  test_accuracy = results["test_acc"]

  # figure out how mamny epochs there were
  epochs = range (len(results["train_loss"]))

  # setup a plot
  plt.figure(figsize = (15,7))

  # plot the loss
  plt.subplot(1,2,1)
  plt.plot(epochs , loss  , label = "train_loss")
  plt.plot(epochs , test_loss , label = "test_loss")
  plt.title("loss")
  plt.xlabel("Epochs")
  plt.legend()

  # plot the accuracy
  plt.subplot(1,2,2)
  plt.plot(epochs , accuracy , label = "train_accuracy")
  plt.plot(epochs , test_accuracy,label = "test_accuracy")
  plt.title("Accuracy")
  plt.xlabel("Epochs")
  plt.legend();

In [ ]:
plot_loss_curves(model_0_results)

## 8. What should an ideal loss curve look like

a loss curve is one of the most helpful ways to torubleshoot a model.

## 9. Model 1 : TinyVGG with data Augmentation .
now lets try another modelling experiment this time using the same model as before with some data augmnetation.


### 9.1 create transform with data augmentation

In [ ]:
# create training transform with TrivialAugmnet
from torchvision import transforms
train_transform_trivial = transforms.Compose([transforms.Resize(size = (64 ,64)),
                                              transforms.TrivialAugmentWide(num_magnitude_bins = 31),
                                              transforms.ToTensor()

                                              ])
test_transform_simple = transforms.Compose([
    transforms.Resize(size = (64, 64)),
    transforms.ToTensor()
])



### 9.2 create train and test Dataset and DataLoader with data augmentation

In [ ]:
# turn image folders into dataset
from torchvision import datasets
train_data_augmented = datasets.ImageFolder(root = train_dir,
                                            transform = train_transform_trivial)

test_data_augmented = datasets.ImageFolder(root = test_dir ,
                                           transform = test_transform_simple)


In [ ]:
# turn  our datasets into dataloader
import os
BATCH_SIZE = 32
NUM_WORKERS = os.cpu_count()

torch.manual_seed(42)
train_dataloader_augmented = DataLoader(dataset =train_data_augmented,
                                        batch_size = BATCH_SIZE,
                                        shuffle = True,
                                        num_workers = NUM_WORKERS)

test_dataloader_augmented = DataLoader(dataset = test_data_simple ,
                                       batch_size = BATCH_SIZE,
                                       shuffle = False,
                                       num_workers = NUM_WORKERS)



### 9.3 construct and train model

* this time we'll be using the same model architrcture except this time we've augmented the training data.

In [ ]:
# create a model_1 and send it to the target device
torch.manual_seed(42)
model_1 = TinyVGG(input_shape =3,
                  hidden_units = 10,
                  output_shape = len(train_data_augmented.classes)).to(device)

model_1

wonderfull now we've got a model and dataloader , lets create a loss function and na optimizer and call upon our train() function to evaluate our model

In [ ]:
# set random seeds
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# set the number of epochs
NUM_EPOCHS = 5

# setup loss function
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params = model_1.parameters(),
                            lr = 0.001)

# start the timer
from timeit import default_timer as timer
start_time = timer()

# train model 1
model_1_results = train(model = model_1,
                        train_dataloader = train_dataloader_augmented,
                        test_dataloader = test_dataloader_augmented,
                        optimizer = optimizer,
                        loss_fn = loss_fn,
                        epochs = NUM_EPOCHS,
                        device = device)

# end the timer and print put how long it took
end_time = timer()

print(f"total training time for model_1 : {end_time-start_time:.3f} seconds")

In [ ]:
model_1_results

### 9.4 Plot the loss curves of model_1


A loss curve helps ur model performace overtime


In [ ]:
plot_loss_curves(model_1_results)

## compare model results

after evaluating our modelling experiments on their own , its important to compare them to each other.

there's few different wanys to do this :     

1. hard coding which we are doing

2. pytorch + tensorboard:  https://docs.pytorch.org/docs/2.12/tensorboard.html

3. weights & biases - https://wandb.ai/site/

4. MLFlow

In [ ]:
import pandas as pd
model_0_df = pd.DataFrame(model_0_results)
model_1_df = pd.DataFrame(model_1_results)
model_0_df


In [ ]:
model_1_df

In [ ]:
# setup a plot
plt.figure(figsize =(15,10))

# get no. o f epochs
epochs = range(len(model_0_df))

# plot train loss
plt.subplot(2, 2, 1)# 2 rows, 2 cols, 1 index no.
plt.plot(epochs , model_0_df["train_loss"] , label="Model 0")
plt.plot(epochs , model_1_df["train_loss"] , label="model 1")
plt.title("train loss")
plt.xlabel("Epochs")
plt.legend()


# plot test loss
plt.subplot(2, 2, 2)# 2 rows, 2 cols, 2 index no.
plt.plot(epochs , model_0_df["test_loss"] , label="Model 0")
plt.plot(epochs , model_1_df["test_loss"] , label="model 1")
plt.title("test loss")
plt.xlabel("Epochs")
plt.legend()



# plot train acc
plt.subplot(2, 2, 3)# 2 rows, 2 cols, 3 index no.
plt.plot(epochs , model_0_df["train_acc"] , label="Model 0")
plt.plot(epochs , model_1_df["train_acc"] , label="model 1")
plt.title("train acc")
plt.xlabel("Epochs")
plt.legend()


# plot test acc
plt.subplot(2, 2, 4)# 2 rows, 2 cols, 4 index no.
plt.plot(epochs , model_0_df["test_acc"] , label="Model 0")
plt.plot(epochs , model_1_df["test_acc"] , label="model 1")
plt.title("test acc")
plt.xlabel("Epochs")
plt.legend()





## 11. Making a rprediction on a custom image

Although we've trained a model on custom data... how do u make a prediction on a sample/image thats's not in either training or testing dataset

In [ ]:
# Download custom image
import requests

# setup custom image path
custom_image_path = data_path / "04-pizza-dad.jpeg"

# download the image if it doesnt exist
if not custom_image_path.is_file():
  with open(custom_image_path, "wb") as f:
    # when downloading from github , need to use the "raw file link"

    request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/images/04-pizza-dad.jpeg")
    print(f"Downloading {custom_image_path}")
    f.write(request.content)

else :
  print(f"{custom_image_path} already exists ,skipping download...")

### 11.1 loading in a custom image with pytorch

we have to make sure our custom image is in the same format as the data our model we trained on.

* in Tensor form with datatype(torch.float32).

* of shape 64*64*3

* on the right device

we can read image into pytorhc using - https://docs.pytorch.org/vision/main/generated/torchvision.io.read_image.html

In [ ]:
import torchvision

# read in custom image
custom_image_unit8 = torchvision.io.read_image(custom_image_path)
print(f"custom image tensor :\n {custom_image_unit8}")
print(f"custom image shape : {custom_image_unit8.shape}")
print(f"custom image datatype : {custom_image_unit8.dtype}")

In [ ]:
plt.imshow(custom_image_unit8.permute(1,2,0))

### 11.2 Making a prediction on a custom image with a trained pytorch model



In [ ]:
# load in the custom image and convert to torch.float32
custom_image = torchvision.io.read_image(str(custom_image_path)).type(torch.float32) / 255.
custom_image

In [ ]:
custom_image.shape

In [ ]:
plt.imshow(custom_image.permute(1,2,0))

In [ ]:
# create transform pipelines to resize image
custom_image_transform = transforms.Compose([
    transforms.Resize(size = (64, 64))

])

# transforms target image
custom_image_transformed = custom_image_transform(custom_image)

# print out the shape
print(f"original shape : {custom_image.shape}")
print(f"transformed shape : {custom_image_transformed.shape}")

In [ ]:
plt.imshow(custom_image_transformed.permute(1,2,0))

In [ ]:
# this will error : image not on the right device
model_1.eval()
with torch.inference_mode():
  custom_image_pred = model_1(custom_image_transformed)

In [ ]:
# this will error :  no batch size
model_1.eval()
with torch.inference_mode():
  custom_image_pred = model_1(custom_image_transformed.to(device))

In [ ]:
custom_image_transformed.shape , custom_image_transformed.unsqueeze(0).shape

In [ ]:
# this should work ? (added a batch size..)
model_1.eval()
with torch.inference_mode():
  custom_image_pred = model_1(custom_image_transformed.unsqueeze(0).to(device))

custom_image_pred

** Note ** , to make a prediction on a custom image we had to :

* Load the image and turn it into a tensor

* Make sure the image was the same datatype as the model(torch.float32)

* make sure the image was the same shape as the data type the model was traine don (3,64,64) with a batch size ....(1,3,64,64).

* make sure the image was on the same device as our model .




In [ ]:
# convert logits -> prediction probabilities
custom_image_pred_probs = torch.softmax(custom_image_pred , dim =1)
custom_image_pred_probs

In [ ]:
# convert prediction probabilities -> prediction labels

custom_image_pred_label = torch.argmax(custom_image_pred_probs, dim =1 )
custom_image_pred_label

In [ ]:
class_name[custom_image_pred_label]

### 11.3 putting custom image prediction together : building a function

Ideal outcome:

A function where we pass an image path to and have our model predict on that image and plot the image + prediction .

In [ ]:
def pred_and_plot_image(model: torch.nn.Module,
                        image_path : str,
                        class_name: List[str] = None,
                        transform = None,
                        device = device):

  """
  makes a prediction on a target image with a trained model and plots the image and predictions.
  """
  # load in the image
  target_image = torchvision.io.read_image(str(image_path)).type(torch.float32)


  # deivide the image pixel values by 255 to get them between [0,1]
  target_image = target_image / 255

  # transform if necessary
  if transform:
    target_image = transform(target_image)

  # make sure the model is on the target device
  model.to(device)

  # turn on eval/inference_mode and make an prediction
  model.eval()
  with torch.inference_mode():
    # add an extra dimension to the image (this is the batch dimension ,e.g. our model will predict on batches of 1x images)
    target_image = target_image.unsqueeze(0)

    # make a prediction on the image with an extra dimension
    target_image_pred = model(target_image.to(device)) # make sure the target is on the right device

  # convert logits. -> prediction probabilities
  target_image_pred_probs = torch.softmax(target_image_pred, dim = 1)

  # convert prediciton probbailities -> prediction label
  target_image_pred_label = torch.argmax(target_image_pred_probs, dim =1)

  # plot the image alongside the prediciton and prediction probability
  plt.imshow(target_image.squeeze().permute(1,2,0)) # remove batch dimension and reaarange shape to be HWC

  if class_name:
    title =f"Pred : {class_name[target_image_pred_label.cpu()]} | prob: {target_image_pred_probs.max().cpu():.3f} "
  else :
    title = f"Pred : {target_image_pred_label} | prob:  {target_image_pred_probs.max().cpu():.3f}"

  plt.title(title)
  plt.axis(False)

In [ ]:
# pred on our custom image
pred_and_plot_image(model = model_1,
                    image_path = custom_image_path,
                    class_name = class_name,
                    transform = custom_image_transform,
                    device =device)